# L2 — Chain determinista

> **Varias llamadas al modelo encadenadas donde el programador controla el orden y el flujo.** El modelo responde cuando se le llama — nunca decide qué paso ejecutar a continuación.

## ¿Qué vas a aprender?

- Cómo encadenar varias llamadas al LLM con flujo controlado por código
- Cómo gestionar **estado entre llamadas** (sin frameworks)
- Cómo defenderte de **prompt injection** con etiquetas estructuradas
- Por qué `temperature=0` es clave para clasificación y extracción

## ¿Qué hace este script?

Toma un ticket de soporte en texto libre y lo procesa en **tres pasos**:

```
Texto libre → [Paso 1] Extracción → [Paso 2] Clasificación → [Paso 3] Acción estructurada
```

El estado entre pasos lo gestionamos nosotros: `entities` y `classification` son variables Python normales que pasamos explícitamente al siguiente paso.

## Por qué es **determinista**

Dos razones:

1. **El flujo lo controla el código.** La secuencia está escrita en Python — siempre 1 → 2 → 3, sin excepciones. El modelo no puede saltarse pasos ni cambiar el orden.

2. **`temperature=0` en extracción y clasificación.** El mismo ticket siempre produce el mismo output. El paso 3 usa `temperature=0.2` porque es escritura de texto — un poco de variedad mejora la redacción.

## Conceptos clave

### Temperature por tarea

| Valor | Comportamiento | Cuándo usarlo |
|-------|---------------|---------------|
| `0`   | Determinista, siempre el mismo output | Clasificación, extracción, parsing |
| `0.2` | Casi determinista, ligera variedad | Escritura técnica, resúmenes |
| `0.7+`| Creativo, variable | Brainstorming, copywriting |

### Prompt injection

El equivalente del SQL injection para LLMs. Un usuario malicioso incluye instrucciones en su input intentando sobreescribir el system prompt.

**Defensa que usamos:** envolver el input del usuario en etiquetas `[USER]...[/USER]` e indicar al modelo que ese contenido son **datos**, no instrucciones. Se complementa validando que el output sea JSON válido con el schema esperado.

---
## Setup — Imports y configuración

Cargamos las variables de entorno (incluida `ANTHROPIC_API_KEY`) y creamos el cliente. El modelo `claude-sonnet-4-5` es un buen balance precio/calidad para tareas estructuradas.

> **Requisitos:** `pip install anthropic python-dotenv`

In [ ]:
"""
L2 — Chain determinista: triaje de tickets de soporte técnico
==============================================================
Tres pasos encadenados usando el SDK oficial de Anthropic.
El flujo lo controla el código, no el modelo.

Paso 1 → Extrae entidades del ticket (componente, síntoma, entorno)
Paso 2 → Clasifica severidad y área
Paso 3 → Genera acción recomendada en formato estructurado

Requisitos:
    pip install anthropic

Variables de entorno:
    ANTHROPIC_API_KEY — tu clave de API de Anthropic
"""

from pathlib import Path
from dotenv import load_dotenv
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / ".env").exists():
        load_dotenv(_p / ".env", override=True)
        break

import os
import json
import anthropic

client = anthropic.Anthropic()  # lee ANTHROPIC_API_KEY del entorno automáticamente
MODEL  = "claude-sonnet-4-5"

## Funciones auxiliares

Dos utilidades que nos van a hacer falta en todos los pasos:

- **`call_claude`**: encapsula la llamada a la API con el patrón system/user que veremos en cada paso
- **`parse_json_safe`**: los LLMs a veces envuelven el JSON en bloques ```json ... ``` — esto los limpia antes de parsear

In [ ]:
# ─────────────────────────────────────────────
# Utilidades
# ─────────────────────────────────────────────

def call_claude(system: str, user: str, temperature: float = 0.0) -> str:
    """
    Llamada al modelo con un system prompt y un mensaje de usuario.

    temperature=0   → determinista, ideal para extracción y clasificación
    temperature=0.2 → ligera variedad, útil para generación de texto
    """
    message = client.messages.create(
        model=MODEL,
        max_tokens=512,
        temperature=temperature,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    return message.content[0].text.strip()


def parse_json_safe(text: str) -> dict:
    """
    Parsea JSON de forma segura.
    Los LLMs a veces envuelven la respuesta en ```json ... ```.
    """
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:-1])
    return json.loads(cleaned)

## Paso 1 — Extracción de entidades

**Objetivo:** convertir un ticket en lenguaje natural en un objeto estructurado con campos que podamos procesar.

**Schema de salida** (lo que esperamos que devuelva el modelo):

```json
{
  "componente": "auth",
  "sintoma": "500 errors on login",
  "entorno": "producción",
  "menciona_datos": false
}
```

**Detalle clave:** `temperature=0` para que la extracción sea determinista. El mismo ticket → la misma extracción siempre.

In [ ]:
# ─────────────────────────────────────────────
# Paso 1 — Extracción de entidades
# ─────────────────────────────────────────────

SYSTEM_EXTRACTOR = """
Eres un extractor de entidades técnicas de tickets de soporte de software.
Analiza el ticket y extrae información estructurada.

Responde SOLO con JSON válido. Sin texto adicional, sin markdown, sin explicaciones.
Schema exacto:
{
  "componente": string,
  "sintoma": string,
  "entorno": "producción" | "staging" | "desarrollo" | "desconocido",
  "menciona_datos": boolean
}
""".strip()


def step1_extract(ticket: str) -> dict:
    """Extrae las entidades clave del ticket en formato estructurado."""
    print("\n[Paso 1] Extrayendo entidades...")

    # Envolvemos el input en etiquetas para separar contexto de instrucción.
    # Defensa básica contra prompt injection: el modelo sabe que
    # lo que hay dentro de [USER][/USER] son datos, no instrucciones.
    raw = call_claude(SYSTEM_EXTRACTOR, f"[USER]\n{ticket}\n[/USER]")

    result = parse_json_safe(raw)
    print(f"  → {result}")
    return result

## Paso 2 — Clasificación de severidad

**Objetivo:** asignar P1/P2/P3/P4 al ticket basándose en el texto original **+ las entidades del paso 1**.

Aquí se ve claramente el patrón de **paso de estado**: `step2_classify` recibe `entities` como parámetro explícito. Nosotros gestionamos el estado, no el modelo.

**Criterios de severidad** (encoded en el system prompt):
- **P1**: sistema caído en producción, pérdida de datos, brecha de seguridad
- **P2**: funcionalidad crítica degradada en producción
- **P3**: bug en producción sin workaround conocido
- **P4**: bug menor, mejora o pregunta

In [ ]:
# ─────────────────────────────────────────────
# Paso 2 — Clasificación de severidad
# ─────────────────────────────────────────────

SYSTEM_CLASSIFIER = """
Eres un clasificador de severidad para tickets de soporte técnico.
Recibirás el ticket original más las entidades ya extraídas en el paso anterior.

Responde SOLO con JSON válido. Sin texto adicional.
Schema exacto:
{
  "severidad": "P1" | "P2" | "P3" | "P4",
  "razon_severidad": string,
  "area": "backend" | "frontend" | "infra" | "datos" | "seguridad",
  "requiere_escalado": boolean
}

Criterios de severidad:
- P1: sistema caído en producción, pérdida de datos, brecha de seguridad
- P2: funcionalidad crítica degradada en producción
- P3: bug en producción sin workaround conocido
- P4: bug menor, mejora o pregunta
""".strip()


def step2_classify(ticket: str, entities: dict) -> dict:
    """
    Clasifica la severidad usando el ticket original y las entidades del paso 1.

    Recibe entities como parámetro explícito — el estado lo gestionamos
    nosotros, no el framework. Esto es lo que define L2 como determinista.
    """
    print("\n[Paso 2] Clasificando severidad...")

    user_msg = f"""
Ticket original:
[USER]
{ticket}
[/USER]

Entidades extraídas (paso anterior):
{json.dumps(entities, ensure_ascii=False, indent=2)}
""".strip()

    raw = call_claude(SYSTEM_CLASSIFIER, user_msg)
    result = parse_json_safe(raw)
    print(f"  → {result}")
    return result

## Paso 3 — Generación de la acción

**Objetivo:** producir el ticket estructurado final listo para ingeniería: título, descripción, pasos siguientes y etiquetas.

**Detalle clave:** este paso usa `temperature=0.2` (no `0`) porque está **redactando texto**. Una pizca de variedad mejora la calidad del lenguaje natural sin perder coherencia.

In [ ]:
# ─────────────────────────────────────────────
# Paso 3 — Generación de acción
# ─────────────────────────────────────────────

SYSTEM_WRITER = """
Eres un agente de soporte técnico senior que estructura tickets para el equipo de ingeniería.
Recibirás el ticket original, las entidades y la clasificación de severidad.

Responde SOLO con JSON válido. Sin texto adicional.
Schema exacto:
{
  "titulo": string,
  "descripcion": string,
  "pasos_siguientes": [string],
  "etiquetas": [string]
}

Restricciones:
- titulo: máximo 80 caracteres, claro y accionable
- pasos_siguientes: entre 2 y 4 acciones concretas
- etiquetas: máximo 5, en kebab-case
""".strip()


def step3_generate_action(ticket: str, entities: dict, classification: dict) -> dict:
    """
    Genera el resumen estructurado listo para el sistema de tickets.
    Usa temperature=0.2 para que el texto generado tenga algo de variedad
    sin perder coherencia — es escritura, no clasificación.
    """
    print("\n[Paso 3] Generando acción estructurada...")

    user_msg = f"""
Ticket original:
[USER]
{ticket}
[/USER]

Entidades:
{json.dumps(entities, ensure_ascii=False, indent=2)}

Clasificación:
{json.dumps(classification, ensure_ascii=False, indent=2)}
""".strip()

    raw = call_claude(SYSTEM_WRITER, user_msg, temperature=0.2)
    result = parse_json_safe(raw)
    print(f"  → {result}")
    return result

## Pipeline completo

`run_chain` orquesta los tres pasos en secuencia. Es donde se ve la naturaleza determinista de L2:

- El orden es **fijo**: siempre 1 → 2 → 3
- Si un paso falla, la excepción sube — no la silenciamos
- El estado se pasa explícitamente entre pasos como variables Python normales

In [ ]:
# ─────────────────────────────────────────────
# Pipeline: la chain completa
# ─────────────────────────────────────────────

def run_chain(ticket: str) -> dict:
    """
    Ejecuta los tres pasos en secuencia.

    El orden es fijo: siempre 1 → 2 → 3.
    El modelo no decide el flujo — eso lo hace este código.
    Si un paso falla, la excepción sube sin silenciarse.
    """
    print("=" * 60)
    print("TICKET:", ticket[:80] + "..." if len(ticket) > 80 else ticket)
    print("=" * 60)

    entities       = step1_extract(ticket)
    classification = step2_classify(ticket, entities)
    action         = step3_generate_action(ticket, entities, classification)

    result = {
        "input":          ticket,
        "entities":       entities,
        "classification": classification,
        "action":         action,
    }

    print("\n" + "=" * 60)
    print("RESULTADO FINAL:")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

## Tickets de ejemplo

Dos casos de prueba que cubren extremos de severidad:

- **`ticket_critical`** → debería clasificarse como **P1** (servicio caído, miles de usuarios afectados)
- **`ticket_minor`** → debería clasificarse como **P3 o P4** (bug visual, sin pérdida de funcionalidad)

In [ ]:
# ─────────────────────────────────────────────
# Ejemplos de uso
# ─────────────────────────────────────────────

ticket_critical = """
Users cannot log in since 09:15 UTC. The authentication service is returning
500 errors on POST /auth/login. All environments affected. We are seeing
database connection timeouts in the logs. Approximately 3,000 users impacted.
""".strip()

ticket_minor = """
On the account settings page, when the user updates their email address,
the confirmation message shows the old email instead of the new one.
The update itself works correctly — it is only a display issue.
Reported in production, low impact.
""".strip()

## Ejecutar

Lanza la chain sobre el ticket crítico. Verás los tres pasos imprimirse en orden con su output:

In [ ]:
run_chain(ticket_critical)

Ahora con el ticket menor — fíjate en cómo cambia la severidad y la acción recomendada:

In [9]:
run_chain(ticket_minor)

  → {'titulo': 'Account settings: confirmation message displays old email after update', 'descripcion': 'When a user updates their email address on the account settings page, the confirmation message incorrectly displays the old email address instead of the new one. The email update itself processes correctly in the backend - this is purely a display issue in the confirmation UI. Reported in production environment with low user impact.', 'pasos_siguientes': ['Inspect the account settings confirmation component to identify where the email value is being sourced', 'Verify the state management flow to ensure the new email is properly passed to the confirmation message', 'Add frontend test coverage for email update confirmation display'], 'etiquetas': ['frontend', 'account-settings', 'display-bug', 'p3', 'production']}

RESULTADO FINAL:
{
  "input": "On the account settings page, when the user updates their email address,\nthe confirmation message shows the old email instead of the new one

{'input': 'On the account settings page, when the user updates their email address,\nthe confirmation message shows the old email instead of the new one.\nThe update itself works correctly — it is only a display issue.\nReported in production, low impact.',
 'entities': {'componente': 'account settings page',
  'sintoma': 'confirmation message shows old email instead of new email after update',
  'entorno': 'producción',
  'menciona_datos': True},
 'classification': {'severidad': 'P3',
  'razon_severidad': 'Bug confirmado en producción que afecta la experiencia del usuario en funcionalidad de cuenta. Aunque el update funciona correctamente y es solo un problema de visualización, genera confusión y afecta la confianza del usuario. No tiene workaround documentado pero el impacto es bajo ya que la funcionalidad subyacente opera correctamente.',
  'area': 'frontend',
  'requiere_escalado': False},
 'action': {'titulo': 'Account settings: confirmation message displays old email after update

## Siguientes pasos → L3

En L2 el flujo lo controla siempre el código y el modelo trabaja "ciego" — solo sabe lo que le pasamos en el prompt.

En **[L3 — Tool use](../L3-TOOL_USE/L3_Tool_Use.ipynb)** el modelo gana la capacidad de **invocar herramientas externas** (consultar el estado de un servicio, buscar tickets similares) — y por primera vez participa en el flujo decidiendo qué tool usar y cuándo.